# Hackathon 1, statistics.

This project illustrates the statistics part of the course LEPL1109. In the first part of the project, you will study the mortality observed in Belgium. In the second part of the project, you try to predict the electricity consumption in Belgium using climate variables.

## Report content

•	Grades are granted to the members whose names are in the Jupyter notebook. If your name doesn’t appear on the top of the notebook, you’ll get a 0, even though you are in a group on Moodle.

•	The jupyter notebook must be compiled with printed results and next submitted via moodle. The absence of compiled results (or non-printed values) leads to a lower grade.

•	Do not comment your results directly into cells of code. Use instead a Markdown cell. 

•	"Dry" code or results not followed by a minimum of analysis / comments will be penalized.


## Report submission

•	Deadline, see moodle website. Submission after the deadline will not be accepted.

•	To submit your report, go to the section “Hackathon” on Moodle and the subsection “Remise Hackathon 1”. You can upload your work there. Once you are sure that it is your final version, click the button “Envoyer le devoir”. It is important that you don’t forget to click on this button ! 

•	Reports that have not been uploaded through Moodle will not be corrected.


## Names and Noma of participants:

Part. 1: AFRAM Patrick - 42672400

Part. 2: HENNE Alexis - 54762100

Part. 3: AMBATIELLO Aleksey - 56122100

Part. 4: KEYEUX Titouan - 66022300

Part. 5: SONNET David - 34572300

Part. 6: VERHAMME Guillaume - 79392400

# Mortality and climate

In this section, we analyze the relationship between mortality, age groups, and seasons in Belgium using publicly available data from Statbel (see https://statbel.fgov.be/en/about-statbel/what-we-do/visualisations/mortality). The dataset contains the number of deaths recorded over time, disaggregated by age group and observed across different seasons. The objective is to study temporal patterns in mortality and assess the role of seasonal effects. 

## 1. Basic statistics

**1.1)** Load the dataset mortality_climate.csv. Convert the variables *season* and *CD_AGEGROUP* into a categorical variable. 

In [ ]:
import math
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np 
import seaborn as sns 
from scipy.stats import poisson, nbinom, t, wilcoxon 
import statsmodels.api as sm 
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split # sklearn utilisé dans la question 5
from sklearn.metrics import mean_absolute_error, mean_squared_error


# Load DataSet 
mortality_df = pd.read_csv("mortality_climate.csv")

# Convert into categorical variables 
mortality_df['season'] = mortality_df['season'].astype('category')
mortality_df['CD_AGEGROUP'] = mortality_df['CD_AGEGROUP'].astype('category')

# age order 
age_order = ['0-24', '25-44', '45-64', '65-74', '75-84', '85+']
mortality_df['CD_AGEGROUP'] = pd.Categorical(mortality_df['CD_AGEGROUP'], categories=age_order, ordered=True)

# Summary 
mortality_df.info()

**1.2)** Plot the evolution of the number of deaths (*MS_NUM_DEATH*) with respect to age group. What do you observe?

In [ ]:

# plot number of death with respect to age_group 
mortality_df.groupby('CD_AGEGROUP')['MS_NUM_DEATH'].sum().plot(
    kind='bar',
    figsize=(10,6), 
    color='steelblue'
) # on aggrege MS_NUM_DEATH pour chaque groupe d'age 

plt.title("Nombre total de décès par tranche d'âge")
plt.xlabel("Tranche d'âge")
plt.ylabel("Nombre de décès")
plt.show()

Comments:

*On observe que le nombre de décès augmente significativement avec l'âge.*

**1.3)** Do a scatter plot of the number of deaths (*MS_NUM_DEATH*) by age group.

In [ ]:
# Code here 
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=mortality_df, 
    x='CD_AGEGROUP', 
    y='MS_NUM_DEATH', 
    color='lightgray', 
)

sns.stripplot(
    data=mortality_df, 
    x='CD_AGEGROUP', 
    y='MS_NUM_DEATH', 
    jitter=0.2, 
    alpha=0.4, 
    color='crimson', 
    size=2
)

plt.title("Distribution et statistiques des décès par tranche d'âge")
plt.xlabel("Tranche d'âge")
plt.ylabel("Nombre de décès")

plt.show()


Comments: 

*En combinant le boxplot et le stripplot on peut observer à la fois les statistiques descriptives ainsi que la dispersion des observations. 
On constate que l'étalement des données augmentent avec l'âge, les groupes les plus âgés présentent une plus grande amplitude de valeurs.* 

## 2. Fit of distributions

**2.1)** For the 45–64 age group and the winter season only, fit both a Poisson and a Negative Binomial distribution to the variable 'MS_NUM_DEATH' using a maximum likelihood approach. Plot histograms overlaid with the fitted probability mass functions. Compare the log-likelihoods and select the most appropriate distribution.

In [ ]:
# Code here

# on adapte les données 
sub_data_set = mortality_df[(mortality_df['CD_AGEGROUP']== '45-64') & (mortality_df['season'] == 'winter')].copy()
data = sub_data_set['MS_NUM_DEATH'].dropna() # rejette les valeurs manquantes 



# --------------------
# maximum likelyhood 
# --------------------

# A) distribution de poisson 
# lambda = moyenne empirique 
lambda_mle = data.mean()

# log-likekyhood poisson 
log_likelyhood_poisson = np.sum(poisson.logpmf(data, mu=lambda_mle))

# B) Distribution binomiale négative 
model_nb = smf.negativebinomial("MS_NUM_DEATH ~ 1", data=sub_data_set).fit(disp=0)

# extraction des données 
mu_nb = model_nb.predict()[0]
alpha_nb = model_nb.params['alpha'] 

# Adaptation des paramètres pour scipy.stats.nbinom (ia utilisée)
n_param = 1 / alpha_nb
p_param = n_param / (n_param + mu_nb)

# Log-likelihood pour binomiale négative
log_likelyhood_neg_binomial = np.sum(nbinom.logpmf(data, n=n_param, p=p_param))

print(f"--- RÉSULTATS ---")
print(f"Poisson λ_hat = {lambda_mle:.4f}")
print(f"Poisson Log-Likelihood: {log_likelyhood_poisson:.2f}\n")

print(f"Negative Binomial μ = {mu_nb:.4f}, α = {alpha_nb:.4f}")
print(f"Negative Binomial (n={n_param:.4f}, p={p_param:.4f})")
print(f"Negative Binomial Log-Likelihood: {log_likelyhood_neg_binomial:.2f}\n")

if log_likelyhood_neg_binomial > log_likelyhood_poisson:
    print("Verdict : La distribution Binomiale Négative offre une meilleure adéquation (Log-Likelihood plus élevée).")
else:
    print("Verdict : La distribution de Poisson offre une meilleure adéquation.")


# tous les prints du dessus proviennent de l'ia, permet de debuguer et visualiser les résultats proprement 




# -----------
# Graphique 
# -----------
fig, ax = plt.subplots(figsize=(10, 6))

# Histogramme (densité / fréquence)
k_values = np.arange(data.min(), data.max() + 1)
counts, bins = np.histogram(data, bins=np.append(k_values, k_values[-1] + 1) - 0.5, density=True)
ax.bar(k_values, counts, width=0.8, alpha=0.4, color='gray', label='Données empiriques (Winter, 45-64)')

# PMF Poisson
pmf_poisson = poisson.pmf(k_values, mu=lambda_mle)
ax.plot(k_values, pmf_poisson, 'o--', color='dodgerblue', linewidth=2, label=f'Poisson (λ={lambda_mle:.2f})')

# PMF Binomiale Négative
pmf_nb = nbinom.pmf(k_values, n=n_param, p=p_param)
ax.plot(k_values, pmf_nb, 's-', color='crimson', linewidth=2, label=f'Binomiale Négative (n={n_param:.2f}, p={p_param:.2f})')

# Habillage du graphique
ax.set_title("Ajustement des distributions Poisson et Binomiale Négative (MLE)\nTranche 45–64 ans - Hiver", fontsize=13, fontweight='bold')
ax.set_xlabel("Nombre de décès (MS_NUM_DEATH)", fontsize=11)
ax.set_ylabel("Probabilité / Densité", fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# Le dernier graphe a été rendu plus joli par ia aussi, fonctionnait avant mais moins lisible que mtn 



Comments: 

*La loi de Poisson suppose que la variance est égale à la moyenne.*
*On observe sur le graphique que la courbe de poisson sur-évalue le résultat quand on se rapproche du centre.* 
*Tandis que la loi binomiale négative s'adapte mieux aux données grâce à son paramètres de dispersion.* 

Choix du modèle : 

*La log-vraisemblance de la distribution de Poisson est de : -1219,36.* 
*La log-vraisemblance de la distribution binomiale négative est de : -1206,54.*

*La log-vraisemblance de la distribution binomiale négative étant supérieure à celle de la distribution de Poisson, on en conclut que la binomiale négative est plus adaptée, c'est donc celle ci qui est sélectionnée.* 

# 3. Regression

We now aim to assess whether seasonality has an impact on the number of deaths. We consider the following linear model:
$$\log(Y_i) = \beta_0 + \beta_1 X^S_{i1} + \cdots + \beta_3 X^S_{i3} + \beta_4 X^{age}_{i1} + \cdots + \beta_8 X^{age}_{i5}  + \varepsilon_i.$$

where $Y_i$ denotes the number of deaths, $X^S_{ij}$ are dummy variables representing the different seasons, $X^{age}_{ij}$ are dummy variables representing the different age groups, and $\varepsilon_i \sim N(0, \sigma^2)$. 

**3.1)** Why is it more appropriate to use a linear regression on $\log(Y)$ rather than directly on $Y$?

Comments:

*Nous utilisons le logarithme car dans le contexte où l’on souhaite savoir si la saison a un effet sur le nombre de décès, il permet de transformer les multiplications en additions et ainsi de travailler sur une échelle commune. On garantit des prédictions positives pour les décès en utilisant la fonction exponentielle pour revenir à l'échelle précèdente. Un même pourcentage donne des augmentations différentes en nombre de décès, mais cela correspond à la même variation sûr l'échelle du logarithme.*

**3.2)** Perform a linear regression of the number of deaths on the seasonal and group age covariates. Use the `OLS` function from the *statsmodels* package. Make sure to convert the variable *season* and *CD_AGEGROUP* into dummy variables before fitting the model.

Does seasonality have a significant impact on mortality? Is the mortality similar for each age group? 

In [ ]:
# Code here

# Création d'une colonne du nombres de décès en log
X= pd.get_dummies(mortality_df[["season", "CD_AGEGROUP"]], drop_first= True) # Utilisation de l'IA pour le choix du package statsmodels et la fonction dummies 
X=sm.add_constant(X.astype(float))
y = mortality_df["MS_NUM_DEATH"]
# Méthode de régression linéaire
lr = sm.OLS(y,X).fit()

# Affichage tableau
print(lr.summary())

Comments: 

*Oui, la saison impacte la mortalité (ici l'hiver a la plus grande), car toutes les p-values des saisons sont significatives (P < 0,05), inférieures à 0,05. 
Et non, la mortalité n'est pas similaire pour chaque groupe d'âge. La p-value des âges est aussi significative par rapport au premier groupe des 0-24 ans. Le nombre de décès augmente drastiquement avec l'âge,
avec la valeur la plus grande pour les 85 ans et plus.*

## 4. Hypothesis test 

We now investigate whether mortality decreases over time by comparing mortality in summer in 2022 and 2025. To this end, we aim to perform the following hypothesis test:

$$
H_0 : \mu_{2022} = \mu_{2025},
$$

$$
H_1 : \mu_{2022} \neq \mu_{2025}.
$$

**4.1)** Create two subsets of the original dataset corresponding to the years 2022 and 2025, respectively.

In [ ]:
# Code here
sub22 = mortality_df[("2022-12-31" >= mortality_df["date"]) & (mortality_df["date"] >= "2022-01-01")]
sub25 = mortality_df[("2025-12-31" >= mortality_df["date"]) & (mortality_df["date"] >= "2025-01-01")]

print(f"\033[1;34m 2022 subset is \033[0;0m \n{sub22},\n\n \033[1;34m2025 subset is\033[0;0m \n{sub25}")

**4.2)** For the 85+ age group, perform a Student’s t-test to compare mortality between 2022 and 2025 **in summer**. Check whether the assumptions required to apply this test are satisfied.

In [ ]:
# Code here
# Filter over age 85 and summer
sub22_summer_over_85 = sub22[(sub22["CD_AGEGROUP"] == "85+") & (sub22["season"] == "summer")]
sub25_summer_over_85 = sub25[(sub25["CD_AGEGROUP"] == "85+") & (sub25["season"] == "summer")]

print(f"\033[1;34m2022 subset filtered by age > 85+ during summer is\033[0;0m \n{sub22_summer_over_85},\n\n \033[1;34m2025 subset filtered by age > 85+ during summer is\033[0;0m \n{sub25_summer_over_85}\n")

# Find t-value = signal/noise = abs(mean(sub22...) - mean(sub25...))/sqrt(V[sub22...]/n1 + V[sub25...]/n2)
# (if unclear, go seek medical attention immediately)

std_sub22 = sub22_summer_over_85["MS_NUM_DEATH"].std()
std_sub25 = sub25_summer_over_85["MS_NUM_DEATH"].std()

mean_sub22 = sub22_summer_over_85["MS_NUM_DEATH"].mean()
mean_sub25 = sub25_summer_over_85["MS_NUM_DEATH"].mean() 

n1 = sub22_summer_over_85["MS_NUM_DEATH"].count()
n2 = sub25_summer_over_85["MS_NUM_DEATH"].count()

t_value = abs(mean_sub22 - mean_sub25) / math.sqrt((math.pow(std_sub22, 2)) / n1 + (math.pow(std_sub25, 2)) / n2)

alpha = 0.05 # Error margin
dof = n1 + n2 - 2 # Degree of liberty
t_compare = t.isf(alpha, dof)

print("""The t_value for the two subsets above is {0:0.5f}.
The value of the student table with alpha = 0.05 and degree of liberty 94 + 94 - 2 = 186 is {1:0.5f}.
We can see that t_value > t_compare, we REJECT the hypothesis H0""".format(t_value, t_compare))

Comments:

*We reject the null hypothesis H0.
The difference between the subsets is NOT due to chance, there exists a factor influencing the mortality rate of the 85+ range of people
between the summers of the years 2022 and 2025. The two subsets are statistically different.*

#### Assumptions
The assumptions for this test are : normal distribution, independence, similar variance, same number of data points and 20-30+ samples

- Independence... OK (someone cannot die in 2022 and 2025...)
- Normal distribution... OK
- Similar variance... OK (var_sub25 / var_sub22 ~= 1.68 <<< 4)
- Same number of data points... OK (both 94 >>> 20-30+)

EVERY ASSUMPTION IS CORRECT, THE TEST IS ELIGIBLE !!

**4.3)** Explain what the Wilcoxon test is. Why could it be relevant in our setting?

Comments: 

*The Wilcoxon test has a similar goal to Student's t_test, except that we compare the median of the subsets rather than the mean. The main difference are that it's not parametric, the data does not have to be normally distributed. It assumes a certain symmetry of values around the median.*
*In our context, a Student t_test would be more appropriate because every assumption is fulfilled. If it wasn't, then a non parametric test such as this one would be our alternative.*

**4.4)** For the 85+ age group, perform a Wilcoxon test to compare mortality between 2022 and 2025 **in summer**. What do you conclude?

In [ ]:
# Code here

wilcoxon_85 = wilcoxon(sub22_summer_over_85["MS_NUM_DEATH"], sub25_summer_over_85["MS_NUM_DEATH"])
print("The p_value of the Wilcoxon test for the two subsets of question 4.2 is {0:0.7f}".format(wilcoxon_85.pvalue))


Comments:

*The p_value (0.0017383) is lower than the chosen alpha (0.05). Thus, we reject the null hypothesis H0 because there is a significant difference between the medians of both the samples.*

**4.5)** Perform the Wilcoxon test but now for the 25–44 age group and the **summer**. What do you conclude in this case?

In [ ]:
# Code here 

sub22_summer_25_44 = sub22[(sub22["CD_AGEGROUP"] == "25-44") & (sub22["season"] == "summer")]
sub25_summer_25_44 = sub25[(sub25["CD_AGEGROUP"] == "25-44") & (sub25["season"] == "summer")]

wilcoxon_25_44 = wilcoxon(sub22_summer_25_44["MS_NUM_DEATH"], sub25_summer_25_44["MS_NUM_DEATH"])
print(f"""\033[1;34m 2022 subset filtered by age between 25 and 44 during summer is \033[0;0m \n{sub22_summer_25_44}, \n\n \033[1;34m 2025 subset filtered by age between 25 and 44 during summer is \033[0;0m \n{sub25_summer_25_44}\n""")
print("The p_value of the Wilcoxon test for the two subsets is {0:0.7f}".format(wilcoxon_25_44.pvalue))


Comments: 

*The p_value (0.6369235) is *higher* than the chosen alpha (0.05). Thus, we accept the null hypothesis H0 because there is no significant difference between the two medians of both the samples.*

# Electricity Consumption and Climate

In this section, we aim to predict electricity consumption using past consumption values and climatic variables. We rely on data provided by ELIA (see https://opendata.elia.be/explore/dataset/ods003/information/?sort=datetime), which records total electricity consumption in Belgium at a 15-minute frequency, and meteorological data from the Royal Meteorological Institute (IRM) (see https://www.meteo.be/fr/services/donnees), which provides daily average temperature observations over Belgium.

The goal is to build a predictive model for electricity demand based on its recent history and weather conditions. From a practical perspective, such forecasting is highly relevant for energy providers and grid operators, as accurately anticipating electricity demand one week ahead helps optimize production planning, ensure grid stability, and reduce operational costs.

## 5. Forecasting 

The goal is to predict electricity consumption at time $t$ using its past values and climate features. Let $Y_t$ denote electricity consumption at time $t$. We consider the following regression model:

$$Y_t = \beta_0 + \beta_1 Y_{t-7Days} + \beta_2 \bar{T}_{t-7Days} + \beta_3 \bar{T}_{t-7Days}^2 + \varepsilon_t,$$

where $Y_{t-7Days}$ denotes electricity consumption observed 7 days earlier, $\bar{T}_{t-7Days}$ is the average temperature observed 7 days earlier$^1$, and $\varepsilon_t \sim \mathcal{N}(0,\sigma^2)$ is an error term.

$^1$ The average temperature is measured at a daily frequency; therefore, $\bar{T}_{t-7Days}$ is constant for all time points $t$ within the same day.

**5.1)** Load the dataset *electricity_climate.csv* and construct the dataset required to perform the regression.

**Warning:** the data are observed at 15-minute intervals.

To build the regression dataset, each row must contain:
- electricity consumption at time $t$: $Y_t$,
- electricity consumption at time $t - 7$ days: $Y_{t-7Days}$,
- temperature at time $t - 7$ days: $\bar{T}_{t-7Days}$,
- the squared temperature at time $t - 7$ days: $\bar{T}_{t-7Days}^2$.

In [ ]:
# Code here 

electricity_df = pd.read_csv("electricity_climate.csv", parse_dates=['Datetime'])
electricity_df = electricity_df.sort_values('Datetime')

shift7 = 7 * 24 * 4 # 7 jours = 672 observations de 15mins

electricity_df['Y_t'] = electricity_df['Consumption']
electricity_df['Y_t-7Days'] = electricity_df['Consumption'].shift(shift7)
electricity_df['T_t-7Days'] = electricity_df['temp_avg'].shift(shift7)
electricity_df['T_t-7Days_sq'] = electricity_df['temp_avg_sq'].shift(shift7)

regression_df = electricity_df.dropna(subset=['Y_t-7Days', 'T_t-7Days', 'T_t-7Days_sq'])

print(regression_df[['Datetime', 'Y_t', 'Y_t-7Days', 'T_t-7Days', 'T_t-7Days_sq']].head())

**5.2)** Split the dataset into a training set (first 75%) and a test set (last 25%).  Use the `OLS` function from the *statsmodels* package to fit linear regression model using only $Y_{t-7Days}$ as explanatory variable.

Analyze the regression output, discuss the statistical significance and performance of the model.

In [ ]:
# Code here 

regression_df = electricity_df.dropna()

x = regression_df[['Y_t-7Days']]
y = regression_df['Y_t']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, shuffle=False)

x_train_sm = sm.add_constant(x_train)
x_test_sm = sm.add_constant(x_test)

model_1 = sm.OLS(y_train, x_train_sm).fit()

print(model_1.summary())

y_pred_1 = model_1.predict(x_test_sm)
mae_1 = mean_absolute_error(y_test, y_pred_1)
rmse_1 = np.sqrt(mean_squared_error(y_test, y_pred_1))

print(f"MAE: {mae_1: .2f}")
print(f"RMSE: {rmse_1: .2f}")


Comments:

*Le coefficient du modèle est proche de 1 ce qui indique une forte correlation entre la consommation à un instant t et celle des 7 jours passés. Pour ce qui est de R², à 68.8%, il indique que le modèle explique bien une bonne partie de la variance de Y_t, mais il reste de la marge, et donc une possibilité d'amélioration.*
*Ensuite, pour p-value, la variable est statistiquement significative et ne modifie donc pas le modèle. Et finalement, pour nos MAE et RMSE, on a des valeurs faibles, démontrant une bonne performance, par rapport à l'échelle de Consumption et a donc une bonne performance prédictive.*

**5.3)** Repeat the regression by including the climate variables in the model.

Are the climate variables statistically significant in explaining electricity consumption 7 days ahead?

In [ ]:
# Code here

x_climate = regression_df[['Y_t-7Days', 'T_t-7Days', 'T_t-7Days_sq']]
y_climate = regression_df['Y_t']

x_train_cl, x_test_cl, y_train_cl, y_test_cl = train_test_split(x_climate, y_climate, test_size=0.25, shuffle=False)

x_train_cl_sm = sm.add_constant(x_train_cl)
x_test_cl_sm = sm.add_constant(x_test_cl)

model_2 = sm.OLS(y_train_cl, x_train_cl_sm).fit()

print(model_2.summary())

y_pred_2 = model_2.predict(x_test_cl_sm)

mae_2 = mean_absolute_error(y_test_cl, y_pred_2)
rmse_2 = np.sqrt(mean_squared_error(y_test_cl, y_pred_2))

print(f"MAE: {mae_2: .2f}")
print(f"RMSE: {rmse_2: .2f}")

Comments:

*Le modèle explique 68,9% de variabilité de la consommation, similaire au modèle sans l'ajout du climat. Ainsi, l'ajout des variables climatiques n'améliorent que légèrement la prédiction.*

*Comme le montre la valeur du F-statistic et du p-value, le modèle global est statistiquement très significatif.*

*Enfin, on observe que les variables sont fortements corrélées entre elles, ce qui peut donc rendre les coefficients du modèle moins stables.*

*En bref, les erreurs sont légèrement inférieures mais l'amélioration est vraiment très légère.*

**5.4)** Compute the MAE and RMSE on the test set for both models: with and without climate variables.

Does the inclusion of climate variables improve the predictive performance of the model?

In [ ]:
# Code here 

print("--- Comparaison des modèles ---")
print(f"Modèle 1 (sans climat) - MAE: {mae_1:.2f}, RMSE: {rmse_1:.2f}")
print(f"Modèle 2 (avec climat) - MAE: {mae_2:.2f}, RMSE: {rmse_2:.2f}")

mae_improvement = (mae_1 - mae_2) / mae_1 * 100
rmse_improvement = (rmse_1 - rmse_2) / rmse_1 * 100

print(f"\nAmélioration avec le climat :")
print(f"MAE: {mae_improvement:.2f}%")
print(f"RMSE: {rmse_improvement:.2f}%")

Comments: 

*L'ajout des variables climatiques n'améliore que très légèrement les performances du modèle.*

*Les variables climatiques n'apportent que des informations marginales, expliquant ainsi l'absence d'écarts significatif.*

**5.5)** For the model including climate variables, plot the predicted electricity consumption versus the true consumption on the test set. Then, produce a second plot focusing only on the first 7 days of the test set.

In [ ]:
# Code here 

plt.figure(figsize=(14, 6))
plt.plot(y_test_cl.index, y_test_cl, label='Vraie consommation', color='blue', alpha=0.7)
plt.plot(y_test_cl.index, y_pred_2, label='Prédiction (modèle 2)', color='red', linestyle='--')
plt.title("Prédiction vs Réalité (Test Set - Modèle avec climat)")
plt.xlabel("Index temporel")
plt.ylabel("Consommation (MW)")
plt.legend()
plt.grid(True)
plt.show()

#7 premiers jours
first_7_days_indices = y_test_cl.index[:7 * 24 * 4]  # 7 jours * 24h * 4 (15 min)
first_7_days_y_test = y_test_cl.loc[first_7_days_indices]
first_7_days_y_pred = y_pred_2.loc[first_7_days_indices]

plt.figure(figsize=(14, 6))
plt.plot(first_7_days_indices, first_7_days_y_test, label='Vraie consommation', color='blue')
plt.plot(first_7_days_indices, first_7_days_y_pred, label='Prédiction', color='red', linestyle='--')
plt.title("Focus sur les 7 premiers jours du test set")
plt.xlabel("Index temporel")
plt.ylabel("Consommation (MW)")
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Comments:

*Pour le Graphique 1 (ensemble du jeu de test), la consommation prédite suit globalement la consommation réelle : les deux courbes montrent les mêmes variations d'un niveau à l'autre (périodes hautes à environ 8000 - 10000 MW, périodes plus basses autour de 5000 - 6000 MW). On remarque cependant que la courbe réelle est plus étendue que la prédiction : certains pics dépassent 11 000 MW alors que la prédiction reste plus proche de 10 000 MW. Le modèle produit des valeurs plus "lissées" que la réalité et sous-estime les extrêmes. Hypothèse : ce lissage serait lié au coef de Y_t-7Days inférieure à 1 (0,83),  qui ramène les prédictions vers la moyenne.*

*Graphique 2 (7 premiers jours de test, soit 672 points). Ce zoom montre ce que le graphique global ne peut pas car trop dense : la consommation suit un cycle journalier très net, avec sept creux nocturnes (environ 2 500 - 3 500 MW) et des pics en journée (environ 7 000 - 7 500 MW). Le modèle reproduit bien cette forme et la position de ces cycles. Les écarts visibles sont de 3 types : Premièrement, des écarts de niveau, par exemple en début de période où la prédiction est inférieur d'environ 750 MW par rapport à  la realité, et en fin de période où la prédiction reste nettement sous la valeur réelle. Deuxièmement, on a des décalages de forme, par exemple vers l'indice 295 800 où la prédiction présente un pic alors que la conso réelle baisse légèrement. Troisièmement, on a des creux parfois un peu plus profonds dans la prédiction que dans la réalité.*

*Hypothèses possibles concernant ces écarts : les écarts s'expliquent car le modèle prédit Y_t à partir de Y_t-7Days (et de la température d'il y a 7 jours), donc toutes les particularités de la semaine précédente (jour férié, météo différente, événement ponctuel) serait transmise à la prédiction.*

*Conclusion : le modèle avec variables climatiques reproduit bien la forme générale et le cycle journalier, avec un très bon accord sur la première semaine (R^2 = 0,82 sur le zoom de 7 jours). Il est moins précis sur les niveaux et les valeurs extrêmes, ce qui est cohérent avec un R^2 plus modeste sur l'ensemble du tests (0,497). Son accord avec la réalité reste proche de celui du modèle sans climat, ce qui indique que les variables climatiques n'apportent qu'un gain marginal.*